In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# 1. Load data
folder = "/kaggle/input/competitions/anveshan-data-sprint-2-nlp"
train = pd.read_csv(f"{folder}/train.csv")
test = pd.read_csv(f"{folder}/test.csv")

# Fill any blank text safely
train["query_text"] = train["query_text"].fillna("").astype(str).str.lower().str.strip()
test["query_text"] = test["query_text"].fillna("").astype(str).str.lower().str.strip()

# 2. Build a fast Word + Character feature extractor (great for Hinglish typos & Devanagari)
vectorizer = FeatureUnion([
    ("word", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2)),
    ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2))
])

X_train = vectorizer.fit_transform(train["query_text"])
y_train = train["target"].values
X_test = vectorizer.transform(test["query_text"])

# 3. Check validation Macro-F1 score (5-fold)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train), dtype=int)

for tr_idx, val_idx in skf.split(X_train, y_train):
    clf = LinearSVC(C=1.0, class_weight="balanced", random_state=42)
    clf.fit(X_train[tr_idx], y_train[tr_idx])
    oof_preds[val_idx] = clf.predict(X_train[val_idx])

cv_macro_f1 = f1_score(y_train, oof_preds, average="macro")
print(f"5-Fold Validation Macro-F1: {cv_macro_f1:.5f} (Organizer Benchmark: 0.52000)")

# 4. Train on 100% of training data and predict test set
final_clf = LinearSVC(C=1.0, class_weight="balanced", random_state=42)
final_clf.fit(X_train, y_train)
test_preds = final_clf.predict(X_test)

# 5. Save submission.csv
sub = pd.DataFrame({
    "example_id": test["example_id"],
    "target": test_preds
})
sub.to_csv("submission.csv", index=False)
print("\nsubmission.csv saved! Shape:", sub.shape)
print(sub.head(5))



5-Fold Validation Macro-F1: 0.77190 (Organizer Benchmark: 0.52000)

submission.csv saved! Shape: (4912, 2)
   example_id  target
0      900001      21
1      900002      38
2      900003      43
3      900004      44
4      900005      20
